In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import multiprocessing

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>
  data = fetch_version_info()


In [10]:
multiprocessing.set_start_method("spawn", force=True)

# ─────────────────────────────────────────────
# 3. CONFIG
# ─────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 1
BATCH_SIZE = 4
LR = 3e-4
IMG_SIZE = 380

TRAIN_CSV = "/Users/anilkumarazad/MyFiles/Project/Diabetic Retinopathy Detection/Data/train_1.csv"

TRAIN_IMGS = "/Users/anilkumarazad/MyFiles/Project/Diabetic Retinopathy Detection/Data/train_images"

print(f"\nUsing device: {DEVICE}")


Using device: cpu


In [11]:
def preprocess_fundus(img_path, img_size=IMG_SIZE):

    if not os.path.exists(img_path):
        print(f"\n❌ Missing image: {img_path}")
        return None

    img = cv2.imread(img_path)

    if img is None:
        print(f"\n❌ Failed to read image: {img_path}")
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Ben Graham preprocessing
    img = cv2.addWeighted(
        img,
        4,
        cv2.GaussianBlur(img, (0, 0), img_size // 30),
        -4,
        128
    )

    return img

In [12]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.1,
        scale_limit=0.15,
        rotate_limit=30,
        p=0.5
    ),

    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        p=0.4
    ),

    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

    ToTensorV2()
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [13]:
class RetinopathyDataset(Dataset):

    def __init__(self, df, img_dir, transform=None):

        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_id = row["id_code"]
        label = row["diagnosis"]

        img_path = os.path.join(
            self.img_dir,
            image_id + ".png"
        )

        img = preprocess_fundus(img_path)

        # Handle broken images
        if img is None:

            img = np.zeros(
                (IMG_SIZE, IMG_SIZE, 3),
                dtype=np.uint8
            )

        if self.transform:
            img = self.transform(image=img)["image"]

        label = torch.tensor(label, dtype=torch.long)

        return img, label

In [14]:
class RetinopathyModel(nn.Module):

    def __init__(self, num_classes=5, pretrained=True):

        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b3",
            pretrained=pretrained,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.head = nn.Sequential(

            nn.Dropout(0.3),

            nn.Linear(in_features, 256),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        features = self.backbone(x)

        return self.head(features)

In [15]:
def train_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    correct = 0

    loop = tqdm(loader, desc="Training")

    for batch_idx, (imgs, labels) in enumerate(loop):

        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        preds = model(imgs)

        loss = criterion(preds, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        correct += (preds.argmax(1) == labels).sum().item()

        loop.set_postfix(
            loss=loss.item(),
            acc=correct / ((batch_idx + 1) * BATCH_SIZE)
        )

    return total_loss / len(loader), correct / len(loader.dataset)

In [16]:
def val_epoch(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0

    all_preds = []
    all_labels = []

    loop = tqdm(loader, desc="Validation")

    with torch.no_grad():

        for imgs, labels in loop:

            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            preds = model(imgs)

            loss = criterion(preds, labels)

            total_loss += loss.item()

            predicted = preds.argmax(1)

            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    kappa = cohen_kappa_score(
        all_labels,
        all_preds,
        weights="quadratic"
    )

    return (
        total_loss / len(loader),
        correct / len(loader.dataset),
        kappa
    )

In [17]:
if __name__ == "__main__":

    print("\nLoading CSV...")

    df = pd.read_csv(TRAIN_CSV)

    print(f"\nDataset size: {len(df)} images")

    print(
        f"\nGrade distribution:\n"
        f"{df['diagnosis'].value_counts().sort_index()}\n"
    )

    # Split dataset
    train_df, val_df = train_test_split(
        df,
        test_size=0.15,
        stratify=df["diagnosis"],
        random_state=42
    )

    # Datasets
    train_ds = RetinopathyDataset(
        train_df,
        TRAIN_IMGS,
        transform=train_transform
    )

    val_ds = RetinopathyDataset(
        val_df,
        TRAIN_IMGS,
        transform=val_transform
    )

    # DataLoaders
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    print("\nChecking one batch...")

    try:
        sample_imgs, sample_labels = next(iter(train_loader))

        print("✅ DataLoader working")
        print("Batch shape:", sample_imgs.shape)

    except Exception as e:

        print("\n❌ DataLoader failed")
        print(e)

        exit()

    # Class weights
    class_counts = (
        df["diagnosis"]
        .value_counts()
        .sort_index()
        .values
    )

    class_weights = torch.tensor(
        1.0 / class_counts,
        dtype=torch.float
    ).to(DEVICE)

    print("\nBuilding model...")

    model = RetinopathyModel(
        pretrained=True
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    # Training
    best_kappa = 0.0

    print("\n🚀 Starting training...\n")

    for epoch in range(1, EPOCHS + 1):

        print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion
        )

        val_loss, val_acc, val_kappa = val_epoch(
            model,
            val_loader,
            criterion
        )

        scheduler.step()

        print(
            f"\nEpoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"QWK: {val_kappa:.4f}"
        )

        if val_kappa > best_kappa:

            best_kappa = val_kappa

            torch.save(
                model.state_dict(),
                "best_model.pth"
            )

            print(
                f"\n✅ Best model saved "
                f"(QWK: {val_kappa:.4f})"
            )

    print("\n🎉 Training complete")

    print(f"Best QWK: {best_kappa:.4f}")

    print("\nModel saved as:")
    print("best_model.pth")


Loading CSV...

Dataset size: 2930 images

Grade distribution:
diagnosis
0    1434
1     300
2     808
3     154
4     234
Name: count, dtype: int64


Checking one batch...
✅ DataLoader working
Batch shape: torch.Size([4, 3, 380, 380])

Building model...

🚀 Starting training...


========== Epoch 1/1 ==========


Validation: 100%|██████████| 110/110 [01:24<00:00,  1.30it/s]


Epoch 01/1 | Train Loss: 1.2171 | Train Acc: 0.6426 | Val Loss: 0.9274 | Val Acc: 0.7432 | QWK: 0.7988

✅ Best model saved (QWK: 0.7988)

🎉 Training complete
Best QWK: 0.7988

Model saved as:
best_model.pth
